# Linked Lists & Bit Manipulation — Technical Reference

## Quick Index

| Technique | When to use | Problems |
| :--- | :--- | :--- |
| Dummy Node | Simplify insert/delete at head; avoid null checks | 21, 82, 203 |
| In-place Reversal | Reverse full list or sublist without extra space | 206, 92, 25 |
| Fast & Slow Pointers | Cycle detection, find middle, kth from end | 141, 142, 876 |
| Merge Two Sorted Lists | Combine two sorted lists in O(n + m) | 21, 23 |
| Find Intersection | Two lists meet at a node — equalize lengths | 160 |
| XOR tricks | Find single number, swap without temp | 136, 260 |
| AND tricks | Check bit, clear lowest set bit, power of two | 191, 231, 338 |
| Bit masking | Enumerate all subsets of a set | 78, 90 |
| Brian Kernighan | Count set bits in O(number of set bits) | 191, 338 |

---
## Dummy Node

Create a throwaway node before the head. Eliminates special cases for operations on the head node — the real head is always `dummy.next`.

In [ ]:
# Template — dummy node
dummy = ListNode(0)
dummy.next = head
cur = dummy
# ... manipulate list using cur ...
return dummy.next               # real head may have changed


# LC 82 — Remove Duplicates from Sorted List II
# Remove all nodes whose value appears more than once
def deleteDuplicates(head):
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        cur = prev.next
        if cur.next and cur.val == cur.next.val:
            while cur.next and cur.val == cur.next.val:
                cur = cur.next          # skip all nodes with this value
            prev.next = cur.next        # bypass the duplicate group
        else:
            prev = prev.next
    return dummy.next

Problems: LC 21, LC 82, LC 203

---
## In-place Reversal

Reverse a linked list using three pointers: `prev`, `cur`, `next_node`. No extra space. For sublist reversal, locate the sublist boundaries first using a dummy node.

In [ ]:
# LC 206 — Reverse Linked List
def reverseList(head):
    prev, cur = None, head
    while cur:
        next_node = cur.next    # save next before overwriting
        cur.next  = prev        # reverse pointer
        prev      = cur         # advance prev
        cur       = next_node   # advance cur
    return prev                 # prev is new head


# LC 92 — Reverse Linked List II (reverse sublist from left to right)
def reverseBetween(head, left, right):
    dummy = ListNode(0, head)
    prev  = dummy
    for _ in range(left - 1):      # move prev to node just before sublist
        prev = prev.next

    cur = prev.next
    for _ in range(right - left):  # reverse (right - left) times
        next_node    = cur.next
        cur.next     = next_node.next
        next_node.next = prev.next
        prev.next    = next_node
    return dummy.next

Problems: LC 206, LC 92, LC 25

---
## Fast & Slow Pointers

**Full explanation and worked examples in:** Sliding Window & Two Pointers Study Guide — Part 1, Fast & Slow Pointers section.

`fast` moves two steps per iteration, `slow` moves one. If a cycle exists they meet inside it. When `fast` reaches null, `slow` is at the middle.

In [ ]:
# Cycle detection
def hasCycle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow == fast: return True
    return False

# Find middle (slow is at middle when fast hits end)
def findMiddle(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

# Find kth node from end (advance fast k steps first)
def kthFromEnd(head, k):
    slow = fast = head
    for _ in range(k):
        fast = fast.next        # fast is k steps ahead
    while fast:
        slow = slow.next
        fast = fast.next
    return slow                 # when fast hits null, slow is k from end

Problems: LC 141, LC 142, LC 876, LC 19

---
## Merge Two Sorted Lists

Use a dummy node as the anchor. At each step, append the smaller of the two current nodes. After one list is exhausted, attach the remainder of the other.

In [ ]:
# LC 21 — Merge Two Sorted Lists
def mergeTwoLists(l1, l2):
    dummy = cur = ListNode(0)
    while l1 and l2:
        if l1.val <= l2.val:
            cur.next = l1
            l1 = l1.next
        else:
            cur.next = l2
            l2 = l2.next
        cur = cur.next
    cur.next = l1 or l2         # attach remaining nodes
    return dummy.next

Problems: LC 21, LC 23 (K-way merge — see Heap cheat sheet)

---
## Find Intersection

Two pointers walk both lists. When one reaches the end, redirect it to the head of the other list. After at most `len(A) + len(B)` steps both pointers are equidistant from the intersection — they meet at it, or both hit null together.

In [ ]:
# LC 160 — Intersection of Two Linked Lists
def getIntersectionNode(headA, headB):
    a, b = headA, headB
    while a != b:
        a = a.next if a else headB  # redirect to other list's head at end
        b = b.next if b else headA
    return a                        # intersection node, or None if no intersection

Problems: LC 160

---
## XOR Tricks

`a ^ a = 0` and `a ^ 0 = a`. XOR is commutative and associative — order doesn't matter. Use to find an element that appears an odd number of times, or to swap two values without a temp variable.

In [ ]:
# Swap without temp
a ^= b
b ^= a
a ^= b

# LC 136 — Single Number
# All numbers appear twice except one — XOR cancels pairs
def singleNumber(nums):
    return __import__('functools').reduce(lambda a, b: a ^ b, nums)

# LC 260 — Single Number III (two unique numbers)
# XOR all → result = a ^ b. Find any set bit → use as mask to separate a and b
def singleNumberIII(nums):
    xor = 0
    for n in nums: xor ^= n
    diff_bit = xor & (-xor)             # isolate rightmost set bit
    a = 0
    for n in nums:
        if n & diff_bit: a ^= n         # XOR only numbers with this bit set
    return [a, xor ^ a]

Problems: LC 136, LC 137, LC 260

---
## AND Tricks

Common bitwise operations using AND. Each trick is a one-liner.

In [ ]:
# Check if bit i is set
is_set = (n >> i) & 1

# Check odd / even
is_odd  = n & 1             # 1 if odd, 0 if even

# Clear the lowest set bit  — used in Brian Kernighan
n = n & (n - 1)

# Isolate the lowest set bit
lowest = n & (-n)

# Power of two check — exactly one bit set
is_power_of_two = n > 0 and (n & (n - 1)) == 0

# Set bit i
n |= (1 << i)

# Clear bit i
n &= ~(1 << i)

# Toggle bit i
n ^= (1 << i)

Problems: LC 231, LC 342, LC 191

---
## Bit Masking — Subset Enumeration

Represent each subset of `n` elements as an integer bitmask of `n` bits. Bit `i` is set if element `i` is included. Iterate from `0` to `2^n - 1` to enumerate all `2^n` subsets.

In [ ]:
# Enumerate all subsets of nums using bitmask
def subsets_bitmask(nums):
    n = len(nums)
    result = []
    for mask in range(1 << n):          # 1 << n = 2^n
        subset = []
        for i in range(n):
            if mask & (1 << i):         # bit i is set → include nums[i]
                subset.append(nums[i])
        result.append(subset)
    return result


# Enumerate all subsets of a bitmask (iterate over sub-masks)
# Used in DP on subsets problems
def enumerate_submasks(mask):
    sub = mask
    while sub > 0:
        # process sub
        sub = (sub - 1) & mask          # next smaller submask

Problems: LC 78, LC 90, LC 1986

---
## Brian Kernighan — Count Set Bits

`n & (n-1)` clears the lowest set bit. Repeat until `n == 0` — the number of iterations equals the number of set bits. Faster than checking each bit when the number is sparse.

In [ ]:
# Count set bits — O(number of set bits)
def countBits(n):
    count = 0
    while n:
        n &= n - 1                      # clear lowest set bit
        count += 1
    return count


# LC 338 — Counting Bits (count set bits for all numbers 0..n)
# DP: bits[i] = bits[i >> 1] + (i & 1)
# i >> 1 shifts right by 1 (same as i // 2); add 1 if i is odd
def countBitsDP(n):
    bits = [0] * (n + 1)
    for i in range(1, n + 1):
        bits[i] = bits[i >> 1] + (i & 1)
    return bits

Problems: LC 191, LC 338, LC 461